In [ ]:
import sys
import os
from pathlib import Path
import importlib
import json

import numpy as np
import matplotlib.pyplot as plt
import torch


# Optional but very useful in notebooks
%load_ext autoreload
%autoreload 2

repo_root = Path("/home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.rl.evaluate_cql_alpha_sweep as eval_cql
import src.rl.evaluate_terminal_reward_models as etm
import src.rl.evaluate_bc_final_seeds as eval_bc



In [ ]:
cohort = "picu"
# health_cohort = "sepsis_picu"
health_cohort = "picu" # dynamic deephit trained on all picu patients, not just sepsis patients

if cohort == "picu":    
    bc_dir = "/home/jake/Documents/AI_Clinician/models/rl_models/choc/bc_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset"
    print(f"Using bc_dir: {bc_dir}")
    dynamic_deephit_model_dir = "/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model"
    print(f"Using dynamic_deephit_model_dir: {dynamic_deephit_model_dir}")


else:
    raise ValueError(f"Unknown cohort: {cohort}")

# first, load the dynamic deephit hyperparameters so we can get the state dimension
dynamic_deephit_dir = eval_cql.get_model_dirs(dynamic_deephit_model_dir)[0]
print(f"Dynamic DeepHit model directory: {dynamic_deephit_dir}")
# load results.json from dynamic_deephit_dir
with open(os.path.join(dynamic_deephit_dir, "results.json"), "r") as f:
    dynamic_deephit_results = json.load(f)
state_dim = dynamic_deephit_results['hyperparameters']['d_model']
print(f"State dimension from dynamic deephit hyperparameters: {state_dim}")

# find folder inside bc_dir that ends "with_seeds"
bc_with_seeds_dir = None
for folder in os.listdir(bc_dir):
    if folder.endswith("with_seeds"):
        bc_with_seeds_dir = os.path.join(bc_dir, folder)
        break

if bc_with_seeds_dir is None:
    raise ValueError("Could not find folder in bc_dir that ends with 'with_seeds'")

print(f"BC with seeds directory: {bc_with_seeds_dir}")

bc_model_dirs = etm.get_model_dirs(bc_with_seeds_dir)

print(f"BC model directories: {bc_model_dirs}")

bc_models = []
for model_dir in bc_model_dirs:
    # load health score sarsa model
    actor, config_manager, hyperparams = eval_bc.load_actor(reward_type='health_score', model_dir=model_dir, state_dim=state_dim)
   
